# 🧠 SpikeLogBERT — Training Pipeline

Spiking Neural Network-based log parser using Spikformer architecture.

**Pipeline**: Raw Log → BERTTokenizer → Spikformer (SNN) → Template ID

---

## Stages
1. Setup: Clone repo + install deps
2. Data: Download HDFS + preprocess
3. Stage 1: Fine-tune BERT teacher
4. Stage 2: Knowledge distillation → SpikeLogBERT
5. Stage 2b (Optional): Direct training
6. Stage 3: Evaluation

## 0. GPU Check

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## 1. Setup: Clone repo + Install dependencies

In [ ]:
# Clone repo
!git clone https://github.com/thuanbui1309/spikeLogBert.git
%cd spikeLogBert

In [ ]:
# Install dependencies (Colab already has torch, transformers)
!pip install spikingjelly scikit-learn pandas tqdm pyyaml -q

In [ ]:
# Verify imports
import torch
import transformers
import spikingjelly
from model import SpikeLogBERT
print(f"torch {torch.__version__}, transformers {transformers.__version__}")
print("✅ All imports OK")

## 2. Data: Download HDFS + Preprocess

In [ ]:
# Download HDFS 2k benchmark dataset
!python data/download.py --dataset HDFS --output data/raw

In [ ]:
# Preprocess into train/val/test splits
!python data/dataset.py --input data/raw/HDFS_2k.log_structured.csv --output data/hdfs

In [ ]:
# Quick check
!echo "Train:"; wc -l data/hdfs/train.txt
!echo "Val:"; wc -l data/hdfs/val.txt
!echo "Test:"; wc -l data/hdfs/test.txt
!echo "Labels:"; cat data/hdfs/label_mapping.txt

## 3. Stage 1: Fine-tune BERT Teacher

Fine-tune `bert-base-cased` on log parsing (multi-class classification).
This teacher model provides the baseline accuracy AND will be used for distillation.

In [ ]:
# Get number of templates from label mapping
with open('data/hdfs/label_mapping.txt') as f:
    NUM_LABELS = len(f.readlines())
print(f"Number of templates: {NUM_LABELS}")

In [ ]:
!python train_teacher.py \
    --dataset_dir data/hdfs \
    --label_num {NUM_LABELS} \
    --epochs 4 \
    --batch_size 32 \
    --lr 5e-5

## 4. Stage 2: Knowledge Distillation → SpikeLogBERT

Distill the fine-tuned BERT teacher into the Spikformer student (SNN).

4 losses:
- Embedding loss (MSE)
- Representation loss (MSE per-layer)
- Logit loss (KL divergence)
- CE loss (cross-entropy)

In [ ]:
import os
TEACHER_PATH = "saved_models/teacher/best_teacher_{}classes".format(NUM_LABELS)
assert os.path.exists(TEACHER_PATH), f"Teacher model not found at {TEACHER_PATH}"
print(f"Teacher model: {TEACHER_PATH}")

In [ ]:
!python distill.py \
    --dataset_dir data/hdfs \
    --teacher_model_path {TEACHER_PATH} \
    --label_num {NUM_LABELS} \
    --depths 6 \
    --dim 768 \
    --num_step 16 \
    --epochs 30 \
    --batch_size 16 \
    --lr 5e-4

## 5. Stage 2b (Optional): Direct Training

Train Spikformer directly without knowledge distillation.
Used for comparison — expected to have lower accuracy.

In [ ]:
# Uncomment to run direct training
# !python train_direct.py \
#     --dataset_dir data/hdfs \
#     --label_num {NUM_LABELS} \
#     --epochs 50 \
#     --batch_size 16

## 6. Stage 3: Evaluation

In [ ]:
# Find the best distilled model
import glob
distilled_models = sorted(glob.glob("saved_models/distilled/spikelogbert_*.pth"))
if distilled_models:
    BEST_MODEL = distilled_models[-1]
    print(f"Best model: {BEST_MODEL}")
else:
    print("⚠️ No distilled model found. Run Stage 2 first.")

In [ ]:
!python evaluate.py \
    --model_path {BEST_MODEL} \
    --dataset_dir data/hdfs \
    --label_num {NUM_LABELS} \
    --num_step 16

In [ ]:
# View results
import json
with open('results/eval_test.json') as f:
    results = json.load(f)
print(f"Parsing Accuracy: {results['parsing_accuracy']:.4f}")
print(f"Macro F1: {results['macro_f1']:.4f}")
print(f"Weighted F1: {results['weighted_f1']:.4f}")

## 7. Save Results to Google Drive (Optional)

In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Copy saved models
# !cp -r saved_models/ /content/drive/MyDrive/spikeLogBert/
# !cp -r results/ /content/drive/MyDrive/spikeLogBert/
# print("✅ Saved to Google Drive")